# YOLOv2 anchor 與框參數化

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/09-anchors/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.2.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import torch
import torch.nn.functional as F


def size_iou(sizes, anchors):
    intersection = torch.minimum(sizes[:,None,:],anchors[None,:,:]).prod(-1)
    return intersection/(sizes.prod(-1)[:,None]+anchors.prod(-1)[None,:]-intersection)


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    anchors = torch.tensor([[16.,16.],[8.,8.]])
    box = torch.tensor([8.,12.,24.,28.])
    center = (box[:2]+box[2:])/2
    wh = box[2:]-box[:2]
    grid_xy = torch.floor(center/16).long()
    offsets = center/16-grid_xy
    ious = size_iou(wh[None],anchors)[0]
    best = int(ious.argmax())
    encoded = torch.cat([torch.logit(offsets.clamp(1e-4,1-1e-4)),torch.log(wh/anchors[best])])
    decoded_center = (grid_xy+encoded[:2].sigmoid())*16
    decoded_wh = anchors[best]*encoded[2:].exp()
    decoded = torch.cat([decoded_center-decoded_wh/2,decoded_center+decoded_wh/2])
    assert torch.allclose(decoded,box,atol=.02)
    assert torch.allclose(ious,torch.tensor([1.,.25]))
    assert torch.allclose(encoded[:2],torch.tensor([-9.210240,-1.098612]),atol=1e-5)
    assert torch.allclose(encoded[:2].sigmoid(),torch.tensor([1e-4,.25]),atol=1e-7)
    raw = torch.nn.Parameter(torch.zeros(1,4,4,2,7))
    pos = torch.zeros(1,4,4,2,dtype=torch.bool)
    ignore = torch.zeros_like(pos)
    x,y = grid_xy.tolist()
    pos[0,y,x,best] = True
    for a in range(2):
        if a != best and ious[a] > .2:
            ignore[0,y,x,a] = True
    obj_target = pos.float()
    valid = ~ignore
    optimizer = torch.optim.SGD([raw],lr=.1)
    before = raw.detach().clone()
    # Only the responsible slot regresses; ignore does not enter objectness BCE.
    regression = F.mse_loss(raw[pos][:,:2].sigmoid(), offsets[None])
    regression = regression + F.mse_loss(raw[pos][:,2:4],torch.log(wh/anchors[best])[None])
    objectness = F.binary_cross_entropy_with_logits(raw[...,4][valid],obj_target[valid])
    classification = F.cross_entropy(raw[pos][:,5:],torch.tensor([0]))
    loss = regression+objectness+classification
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    assert raw.grad[ignore].abs().sum() == 0
    assert raw.grad[~pos][...,:4].abs().sum() == 0
    optimizer.step()
    assert not torch.equal(raw,before)
    print('size IoU',ious.tolist(),'best anchor',best,'log wh',encoded[2:].tolist())
    print('ratio offsets',offsets.tolist(),'encoded logits',encoded[:2].tolist())
    print('positive/ignore/negative',int(pos.sum()),int(ignore.sum()),int((~pos&~ignore).sum()))
    print('decoded',decoded.tolist(),'one slot update completed')


if __name__ == '__main__':
    main()


size IoU [1.0, 0.25] best anchor 0 log wh [0.0, 0.0]
ratio offsets [0.0, 0.25] encoded logits [-9.210240364074707, -1.0986123085021973]
positive/ignore/negative 1 1 30
decoded [8.00160026550293, 12.0, 24.00160026550293, 28.0] one slot update completed
